# Notebook 2 — Preprocessing Pipeline (CLAHE + Augmentation + Normalization)

This notebook builds the PyTorch preprocessing pipeline for the glaucoma
classification project. It **consumes** the exact train/val/test split
produced by Notebook 1 (`split_manifest.csv`) -- it does **not** create a new
split, does **not** modify `split_manifest.csv`, and never writes into the
raw dataset. Everything here is read-only with respect to Notebook 1's
outputs; the only new files this notebook writes are preprocessing
diagnostics (figures) and a reusable `preprocessing.py` module for later
notebooks to import.

**Official dataset (approved, from Notebook 1):**
- 2,468 usable images -- 1,925 Normal / 543 Glaucoma
- Train 1,729 / Val 370 / Test 369
- Seed 42, stratified by source x class, same-eye/near-duplicate groups kept
  within a single split, 2 conflicting-label images excluded
- `split_manifest.csv` is the authoritative split definition -- this notebook
  reads it and nothing else decides the split.

## 1. Environment setup and package installation

In [ ]:
import importlib, subprocess, sys

REQUIRED_PACKAGES = ["pandas", "numpy", "Pillow", "matplotlib", "opencv-python", "torch", "torchvision"]

def _ensure(pkg):
    import_name = {"Pillow": "PIL", "opencv-python": "cv2"}.get(pkg, pkg)
    try:
        importlib.import_module(import_name)
        print(f"[ok]      {pkg} already available")
    except ImportError:
        print(f"[install] {pkg} not found, attempting pip install ...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
            print(f"[ok]      {pkg} installed")
        except Exception as e:
            print(f"[warning] could not install {pkg} automatically ({e}). "
                  f"Install it manually if a later cell fails on import.")

for _pkg in REQUIRED_PACKAGES:
    _ensure(_pkg)

import os, sys, json, hashlib, inspect, random, time
from pathlib import Path

import pandas as pd
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

print()
print("Environment ready.")
print("pandas", pd.__version__, "| numpy", np.__version__, "| opencv", cv2.__version__)
print("torch", torch.__version__, "| torchvision", __import__("torchvision").__version__)
print("CUDA available:", torch.cuda.is_available())

## 2. Configuration and reproducibility settings

Same seed (42) as Notebook 1, applied here to `random`, `numpy`, and
`torch` so augmentation sampling and DataLoader shuffling are reproducible
run-to-run.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Reproducibility seed set to {SEED} (random, numpy, torch).")

# --- CLAHE parameters (exactly as specified) ---
CLAHE_CLIP_LIMIT = 2.0
CLAHE_TILE_GRID_SIZE = (8, 8)

# --- Image size parameters ---
RESIZE_SIZE = 256
CROP_SIZE = 224

# --- ImageNet normalization (standard for transfer-learning backbones) ---
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# --- DataLoader parameters ---
BATCH_SIZE = 32
NUM_WORKERS = 0  # 0 is safest for interactive Jupyter/Colab; raise for speed in a script

print("CLAHE_CLIP_LIMIT     :", CLAHE_CLIP_LIMIT)
print("CLAHE_TILE_GRID_SIZE :", CLAHE_TILE_GRID_SIZE)
print("RESIZE_SIZE / CROP_SIZE:", RESIZE_SIZE, "/", CROP_SIZE)
print("IMAGENET_MEAN / STD  :", IMAGENET_MEAN, "/", IMAGENET_STD)
print("BATCH_SIZE / NUM_WORKERS:", BATCH_SIZE, "/", NUM_WORKERS)

## 3. Dataset path configuration

Same portable pattern as Notebook 1: change `RUNTIME_ENV` to switch between
the local Mac and Google Drive/Colab without touching the rest of the
pipeline. `RESULTS_OUT_DIR` is where `split_manifest.csv` lives;
`DATASET_OUT_DIR` is where Notebook 1 materialized the train/val/test
folders; `COMMON_DIR` is a new folder this notebook writes the reusable
`preprocessing.py` module into (for Notebook 3+ to import); `FIGURES_OUT_DIR`
is where diagnostic figures are saved.

In [ ]:
RUNTIME_ENV = "local_mac"   # "local_mac" or "colab_drive"

PATH_PRESETS = {
    "local_mac": {
        "RAW_DATASET_ROOT": os.path.expanduser("~/Downloads/archive"),
        "OUTPUT_ROOT": os.path.expanduser("~/Downloads/GlaucomaProject"),
    },
    "colab_drive": {
        "RAW_DATASET_ROOT": "/content/drive/MyDrive/GlaucomaProject/raw_kaggle_download",
        "OUTPUT_ROOT": "/content/drive/MyDrive/GlaucomaProject",
    },
}

_paths = PATH_PRESETS[RUNTIME_ENV]
RAW_DATASET_ROOT = _paths["RAW_DATASET_ROOT"]
OUTPUT_ROOT = _paths["OUTPUT_ROOT"]

DATASET_OUT_DIR = os.path.join(OUTPUT_ROOT, "dataset")     # written by Notebook 1
RESULTS_OUT_DIR = os.path.join(OUTPUT_ROOT, "results")      # split_manifest.csv lives here
COMMON_DIR = os.path.join(OUTPUT_ROOT, "common")            # this notebook writes preprocessing.py here
FIGURES_OUT_DIR = os.path.join(OUTPUT_ROOT, "figures")      # diagnostic figures

os.makedirs(COMMON_DIR, exist_ok=True)
os.makedirs(FIGURES_OUT_DIR, exist_ok=True)

split_manifest_path = os.path.join(RESULTS_OUT_DIR, "split_manifest.csv")
assert os.path.isfile(split_manifest_path), (
    f"split_manifest.csv not found at {split_manifest_path}. "
    f"Run Notebook 1 first -- Notebook 2 does not create a split."
)
assert os.path.isdir(DATASET_OUT_DIR), (
    f"DATASET_OUT_DIR not found at {DATASET_OUT_DIR}. Run Notebook 1 first."
)

print("RUNTIME_ENV      :", RUNTIME_ENV)
print("RAW_DATASET_ROOT :", RAW_DATASET_ROOT)
print("OUTPUT_ROOT      :", OUTPUT_ROOT)
print("DATASET_OUT_DIR  :", DATASET_OUT_DIR, "(exists)")
print("RESULTS_OUT_DIR  :", RESULTS_OUT_DIR)
print("COMMON_DIR       :", COMMON_DIR)
print("FIGURES_OUT_DIR  :", FIGURES_OUT_DIR)
print("split_manifest.csv found at:", split_manifest_path)

## 4. Load Notebook 1's authoritative split (read-only)

This notebook **consumes** `split_manifest.csv` -- it is loaded once, its
MD5 hash is recorded, and that hash is re-checked at the very end of this
notebook (Section 18) to prove the file was never written to. Notebook 2
must never call `.to_csv()` on this file or otherwise create a new split.

In [ ]:
def file_md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

split_manifest_md5_before = file_md5(split_manifest_path)
print("split_manifest.csv MD5 (recorded at load time):", split_manifest_md5_before)

manifest_df = pd.read_csv(split_manifest_path)
print(f"Loaded split_manifest.csv: {len(manifest_df)} rows, columns: {list(manifest_df.columns)}")

EXPECTED_SPLIT = {
    "total": 2468, "normal": 1925, "glaucoma": 543,
    "train": 1729, "val": 370, "test": 369,
}

actual = {
    "total": len(manifest_df),
    "normal": int((manifest_df["label"] == 0).sum()),
    "glaucoma": int((manifest_df["label"] == 1).sum()),
    "train": int((manifest_df["split"] == "train").sum()),
    "val": int((manifest_df["split"] == "val").sum()),
    "test": int((manifest_df["split"] == "test").sum()),
}

print("\nRegression check against the approved Notebook 1 dataset:")
all_match = True
for k in EXPECTED_SPLIT:
    ok = actual[k] == EXPECTED_SPLIT[k]
    all_match &= ok
    print(f"  [{'PASS' if ok else 'MISMATCH'}] {k:10s}: actual={actual[k]:5d}  expected={EXPECTED_SPLIT[k]:5d}")

print()
print("Overall:", "MATCHES the approved Notebook 1 split." if all_match else
                   "DOES NOT MATCH -- stop and investigate before proceeding.")
assert all_match, "split_manifest.csv does not match the approved Notebook 1 dataset."

## 5. Confirm the raw dataset stays untouched

This notebook never opens a raw-dataset file in write mode and never deletes
or moves anything under `RAW_DATASET_ROOT`. As a concrete, checkable proof
rather than just a promise, the raw dataset's file count and total size are
recorded now and re-checked in Section 18 -- if either changes, something
outside this notebook's intended behavior happened.

In [ ]:
def raw_dataset_fingerprint(root):
    n_files, total_bytes = 0, 0
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            fp = os.path.join(dirpath, fn)
            try:
                total_bytes += os.path.getsize(fp)
                n_files += 1
            except OSError:
                pass
    return n_files, total_bytes

t0 = time.time()
raw_n_files_before, raw_bytes_before = raw_dataset_fingerprint(RAW_DATASET_ROOT)
print(f"RAW_DATASET_ROOT fingerprint (computed in {time.time()-t0:.1f}s):")
print(f"  files: {raw_n_files_before}")
print(f"  bytes: {raw_bytes_before} ({raw_bytes_before/1e9:.2f} GB)")
print("This will be re-checked at the end of the notebook (Section 18) and must be unchanged.")

## 6. CLAHE implementation

CLAHE (Contrast Limited Adaptive Histogram Equalization) is applied **only**
to the luminance channel, exactly as specified:

1. Convert RGB -> YUV.
2. Run CLAHE on the Y (luminance) channel only, with `clipLimit=2.0` and
   `tileGridSize=(8, 8)` -- leaving the U/V (chrominance/color) channels
   untouched avoids introducing color distortion, which would happen if
   CLAHE were applied to each RGB channel independently.
3. Convert YUV back to RGB.

This is implemented as a plain function first (for a quick sanity check
below), then wrapped as a torchvision-transform-compatible class in
Section 8 so it can sit at the front of a `transforms.Compose(...)`
pipeline, operating on PIL images exactly like `Resize`, `RandomCrop`, etc.

In [ ]:
def apply_clahe(img_rgb_array, clip_limit=CLAHE_CLIP_LIMIT, tile_grid_size=CLAHE_TILE_GRID_SIZE):
    """img_rgb_array: HxWx3 uint8 numpy array, RGB order. Returns the same shape/dtype."""
    img_yuv = cv2.cvtColor(img_rgb_array, cv2.COLOR_RGB2YUV)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    img_yuv[:, :, 0] = clahe.apply(img_yuv[:, :, 0])
    img_rgb_out = cv2.cvtColor(img_yuv, cv2.COLOR_YUV2RGB)
    return img_rgb_out

# Quick sanity check on one real image from the training split
_sample_row = manifest_df[manifest_df["split"] == "train"].iloc[0]
_sample_path = os.path.join(DATASET_OUT_DIR, _sample_row["split"], _sample_row["class_name"],
                             f"{_sample_row['source']}_{_sample_row['filename']}")
_sample_img = np.array(Image.open(_sample_path).convert("RGB"))
_sample_clahe = apply_clahe(_sample_img)

print("Sanity check image:", _sample_path)
print("Original shape/dtype :", _sample_img.shape, _sample_img.dtype)
print("CLAHE output shape/dtype:", _sample_clahe.shape, _sample_clahe.dtype)
print("Pixel value range after CLAHE:", _sample_clahe.min(), "-", _sample_clahe.max())
assert _sample_clahe.shape == _sample_img.shape and _sample_clahe.dtype == _sample_img.dtype
print("PASS: CLAHE preserves image shape and dtype.")

## 7. Torchvision-compatible CLAHE transform

Wraps `apply_clahe` in a callable class that accepts and returns a PIL
image, so it can be dropped into `transforms.Compose([...])` alongside the
standard torchvision transforms.

In [ ]:
class CLAHETransform:
    """Applies CLAHE to the Y/luminance channel of an RGB image (RGB -> YUV -> CLAHE(Y) -> RGB).
    Accepts and returns a PIL Image, so it composes with standard torchvision transforms."""

    def __init__(self, clip_limit=CLAHE_CLIP_LIMIT, tile_grid_size=CLAHE_TILE_GRID_SIZE):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, pil_img):
        arr = np.array(pil_img.convert("RGB"))
        arr_clahe = apply_clahe(arr, self.clip_limit, self.tile_grid_size)
        return Image.fromarray(arr_clahe)

    def __repr__(self):
        return f"{self.__class__.__name__}(clip_limit={self.clip_limit}, tile_grid_size={self.tile_grid_size})"

_t = CLAHETransform()
_out = _t(Image.open(_sample_path).convert("RGB"))
print(_t)
print("CLAHETransform output:", _out.size, _out.mode)

## 8. Training transform pipeline

`CLAHE -> Resize(256) -> RandomCrop(224) -> RandomHorizontalFlip(p=0.5) ->
RandomRotation(15) -> ColorJitter(brightness=0.1, contrast=0.1) -> ToTensor
-> Normalize(ImageNet mean/std)`.

**`RandomVerticalFlip` is deliberately not used** (locked-in methodology
decision from the project's methodology review) -- fundus photographs have a
consistent anatomical up/down orientation, so vertically flipping them would
create anatomically implausible training examples. The assertion below is a
regression guard, not just documentation.

In [ ]:
def build_train_transform(clip_limit=CLAHE_CLIP_LIMIT, tile_grid_size=CLAHE_TILE_GRID_SIZE,
                           resize_size=RESIZE_SIZE, crop_size=CROP_SIZE,
                           mean=IMAGENET_MEAN, std=IMAGENET_STD):
    return transforms.Compose([
        CLAHETransform(clip_limit, tile_grid_size),
        transforms.Resize(resize_size),
        transforms.RandomCrop(crop_size),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

train_transform = build_train_transform()
print(train_transform)

assert not any(isinstance(t, transforms.RandomVerticalFlip) for t in train_transform.transforms), \
    "RandomVerticalFlip must not be used -- locked-in methodology decision."
print("\nPASS: no RandomVerticalFlip in the training pipeline.")

## 9. Validation/test transform pipeline

`CLAHE -> Resize(256) -> CenterCrop(224) -> ToTensor -> Normalize(ImageNet
mean/std)`. No random augmentation -- validation and test images must be
evaluated deterministically, and `CenterCrop` (not `RandomCrop`) is used so
every run sees exactly the same 224x224 region.

In [ ]:
def build_eval_transform(clip_limit=CLAHE_CLIP_LIMIT, tile_grid_size=CLAHE_TILE_GRID_SIZE,
                          resize_size=RESIZE_SIZE, crop_size=CROP_SIZE,
                          mean=IMAGENET_MEAN, std=IMAGENET_STD):
    return transforms.Compose([
        CLAHETransform(clip_limit, tile_grid_size),
        transforms.Resize(resize_size),
        transforms.CenterCrop(crop_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

eval_transform = build_eval_transform()
print(eval_transform)

## 10. Custom PyTorch `Dataset`

Reads images from the **materialized** `DATASET_OUT_DIR/{split}/{class}/`
folders Notebook 1 built (not the raw dataset), filtered to one `split` at a
time from `manifest_df`. The materialized filename convention
(`{source}_{filename}`) matches exactly what Notebook 1 used when copying
files, so no path needs to be reconstructed against the raw dataset.

In [ ]:
class GlaucomaDataset(Dataset):
    """PyTorch Dataset over one split ("train"/"val"/"test") of the
    materialized glaucoma dataset. Reads image paths and labels from a
    manifest DataFrame (e.g. Notebook 1's split_manifest.csv) -- it does not
    decide the split itself."""

    def __init__(self, manifest_df, split, dataset_out_dir, transform=None):
        assert split in ("train", "val", "test"), f"Unknown split: {split}"
        self.df = manifest_df[manifest_df["split"] == split].reset_index(drop=True)
        self.dataset_out_dir = dataset_out_dir
        self.transform = transform
        self.split = split

    def __len__(self):
        return len(self.df)

    def _resolve_path(self, row):
        return os.path.join(self.dataset_out_dir, row["split"], row["class_name"],
                             f"{row['source']}_{row['filename']}")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = self._resolve_path(row)
        img = Image.open(path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        label = int(row["label"])
        return img, label

print("GlaucomaDataset defined.")

## 11. Build datasets and DataLoaders

Train uses `train_transform` (with augmentation) and is shuffled; val/test
use `eval_transform` (deterministic) and are not shuffled. A seeded
`torch.Generator` controls the train shuffle order for reproducibility.

In [ ]:
def get_dataloaders(manifest_df, dataset_out_dir, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, seed=SEED):
    train_ds = GlaucomaDataset(manifest_df, "train", dataset_out_dir, transform=build_train_transform())
    val_ds = GlaucomaDataset(manifest_df, "val", dataset_out_dir, transform=build_eval_transform())
    test_ds = GlaucomaDataset(manifest_df, "test", dataset_out_dir, transform=build_eval_transform())

    g = torch.Generator()
    g.manual_seed(seed)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=num_workers, generator=g)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return train_loader, val_loader, test_loader, train_ds, val_ds, test_ds

train_loader, val_loader, test_loader, train_ds, val_ds, test_ds = get_dataloaders(manifest_df, DATASET_OUT_DIR)

print(f"train_ds: {len(train_ds)} images  ({len(train_loader)} batches at batch_size={BATCH_SIZE})")
print(f"val_ds  : {len(val_ds)} images  ({len(val_loader)} batches at batch_size={BATCH_SIZE})")
print(f"test_ds : {len(test_ds)} images  ({len(test_loader)} batches at batch_size={BATCH_SIZE})")

assert len(train_ds) == EXPECTED_SPLIT["train"]
assert len(val_ds) == EXPECTED_SPLIT["val"]
assert len(test_ds) == EXPECTED_SPLIT["test"]
print("\nPASS: dataset sizes match the approved Notebook 1 split exactly.")

## 12. Class weights from the actual class counts

Computed from the **training split's** actual counts (not a hardcoded
literature number) using the standard inverse-frequency "balanced" formula:
`weight_c = n_train / (num_classes * n_c)`. Class weights for a loss
function should reflect the *training* distribution specifically (what the
model will actually be optimized against), not the overall dataset -- but
because the split is stratified, the train-split ratio is expected to
closely match the overall approved ratio (1,925 Normal / 543 Glaucoma,
about 3.545:1), which is verified explicitly below.

In [ ]:
def compute_class_weights(manifest_df, split="train"):
    df = manifest_df[manifest_df["split"] == split]
    counts = df["label"].value_counts().sort_index()  # index 0=Normal, 1=Glaucoma
    n = len(df)
    num_classes = len(counts)
    weights = n / (num_classes * counts)
    return torch.tensor(weights.values, dtype=torch.float32), counts.to_dict()

class_weights, train_counts = compute_class_weights(manifest_df, split="train")

print("Train split class counts:", {("Normal" if k == 0 else "Glaucoma"): v for k, v in train_counts.items()})
print("Computed class weights (index 0=Normal, 1=Glaucoma):", class_weights.tolist())

overall_ratio = 1925 / 543
train_ratio = train_counts[0] / train_counts[1]
print(f"\nOverall Normal:Glaucoma ratio (approved dataset): {overall_ratio:.4f}")
print(f"Train-split Normal:Glaucoma ratio                : {train_ratio:.4f}")
print(f"Difference: {abs(overall_ratio - train_ratio):.4f} "
      f"({'matches closely, as expected from stratification' if abs(overall_ratio - train_ratio) < 0.05 else 'UNEXPECTED DIVERGENCE -- investigate'})")

print("\nUse with e.g. nn.CrossEntropyLoss(weight=class_weights) in Notebook 3+.")

## 13. Verify DataLoader output shape and class labels

Pulls one real batch from each of `train_loader`, `val_loader`, and
`test_loader` and checks: image tensor shape is
`(batch_size, 3, 224, 224)`, dtype is `float32`, label tensor shape is
`(batch_size,)`, and label values are only `{0, 1}`.

In [ ]:
def verify_batch(loader, name, expected_crop=CROP_SIZE):
    images, labels = next(iter(loader))
    print(f"=== {name} ===")
    print(f"  images: shape={tuple(images.shape)} dtype={images.dtype}")
    print(f"  labels: shape={tuple(labels.shape)} dtype={labels.dtype} unique={sorted(labels.unique().tolist())}")
    assert images.ndim == 4 and images.shape[1] == 3, "Expected (B, 3, H, W) images"
    assert images.shape[2] == expected_crop and images.shape[3] == expected_crop, \
        f"Expected {expected_crop}x{expected_crop} crops, got {images.shape[2]}x{images.shape[3]}"
    assert set(labels.unique().tolist()).issubset({0, 1}), "Labels must be in {0, 1}"
    print(f"  PASS: shape and label checks OK for {name}.")
    return images, labels

train_images, train_labels = verify_batch(train_loader, "train_loader")
val_images, val_labels = verify_batch(val_loader, "val_loader")
test_images, test_labels = verify_batch(test_loader, "test_loader")

print()
print("All three loaders verified: image tensors are (B, 3, 224, 224) float32, "
      "label tensors are (B,) with values in {0, 1}.")

## 14. Display before/after CLAHE examples

One example per source dataset (ORIGA, G1020, REFUGE_train, REFUGE_val),
raw image vs. after CLAHE, saved to `FIGURES_OUT_DIR`.

In [ ]:
sources_present = sorted(manifest_df["source"].unique())
example_rows = []
for src in sources_present:
    row = manifest_df[manifest_df["source"] == src].iloc[0]
    example_rows.append(row)

n = len(example_rows)
fig, axes = plt.subplots(n, 2, figsize=(8, 4 * n))
if n == 1:
    axes = axes.reshape(1, 2)

for i, row in enumerate(example_rows):
    path = os.path.join(DATASET_OUT_DIR, row["split"], row["class_name"], f"{row['source']}_{row['filename']}")
    orig = Image.open(path).convert("RGB")
    clahe_img = CLAHETransform()(orig)

    axes[i][0].imshow(orig)
    axes[i][0].set_title(f"{row['source']} / {row['filename']}\nBefore CLAHE ({row['class_name']})", fontsize=10)
    axes[i][0].axis("off")

    axes[i][1].imshow(clahe_img)
    axes[i][1].set_title("After CLAHE (Y-channel, clip=2.0, tile=8x8)", fontsize=10)
    axes[i][1].axis("off")

plt.tight_layout()
clahe_fig_path = os.path.join(FIGURES_OUT_DIR, "notebook2_clahe_before_after.png")
fig.savefig(clahe_fig_path, dpi=130)
plt.show()
print("Saved:", clahe_fig_path)

## 15. Display a denormalized batch after augmentation

Pulls one training batch (with the full training augmentation pipeline
applied) and reverses the ImageNet normalization for display, so the images
show the actual augmented/CLAHE'd appearance a training step would see.

In [ ]:
def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    mean_t = torch.tensor(mean).view(3, 1, 1)
    std_t = torch.tensor(std).view(3, 1, 1)
    return (tensor * std_t + mean_t).clamp(0, 1)

batch_images, batch_labels = next(iter(train_loader))
n_show = min(8, batch_images.shape[0])

fig, axes = plt.subplots(2, (n_show + 1) // 2, figsize=(3 * ((n_show + 1) // 2), 6.5))
axes = axes.flatten()
class_name_map = {0: "Normal", 1: "Glaucoma"}

for i in range(n_show):
    img = denormalize(batch_images[i]).permute(1, 2, 0).numpy()
    axes[i].imshow(img)
    axes[i].set_title(class_name_map[int(batch_labels[i])], fontsize=10)
    axes[i].axis("off")
for i in range(n_show, len(axes)):
    axes[i].axis("off")

fig.suptitle("Denormalized training batch (CLAHE + augmentation applied)", fontsize=12)
plt.tight_layout()
batch_fig_path = os.path.join(FIGURES_OUT_DIR, "notebook2_augmented_batch.png")
fig.savefig(batch_fig_path, dpi=130)
plt.show()
print("Saved:", batch_fig_path)

## 16. Save reusable preprocessing/configuration code

Writes `common/preprocessing.py` using `inspect.getsource()` on the classes
and functions actually defined and tested above -- this guarantees the saved
module is *identical* to what this notebook verified, not a hand-copied,
potentially-drifted duplicate. Notebook 3+ can `sys.path.append` the
`common/` folder and `from preprocessing import ...` instead of redefining
any of this.

In [ ]:
module_lines = [
    "# preprocessing.py -- reusable preprocessing/config module for the glaucoma project",
    "# Auto-generated by Notebook 2 (Preprocessing Pipeline) via inspect.getsource() on",
    "# the live, tested definitions -- do not hand-edit; re-run Notebook 2 to regenerate.",
    "",
    "import os",
    "import cv2",
    "import numpy as np",
    "import torch",
    "from torch.utils.data import Dataset, DataLoader",
    "from torchvision import transforms",
    "from PIL import Image",
    "",
    f"CLAHE_CLIP_LIMIT = {CLAHE_CLIP_LIMIT!r}",
    f"CLAHE_TILE_GRID_SIZE = {CLAHE_TILE_GRID_SIZE!r}",
    f"RESIZE_SIZE = {RESIZE_SIZE!r}",
    f"CROP_SIZE = {CROP_SIZE!r}",
    f"IMAGENET_MEAN = {IMAGENET_MEAN!r}",
    f"IMAGENET_STD = {IMAGENET_STD!r}",
    f"BATCH_SIZE = {BATCH_SIZE!r}",
    f"NUM_WORKERS = {NUM_WORKERS!r}",
    f"SEED = {SEED!r}",
    "",
    inspect.getsource(apply_clahe),
    inspect.getsource(CLAHETransform),
    inspect.getsource(build_train_transform),
    inspect.getsource(build_eval_transform),
    inspect.getsource(GlaucomaDataset),
    inspect.getsource(get_dataloaders),
    inspect.getsource(compute_class_weights),
    inspect.getsource(denormalize),
]
module_source = "\n\n".join(module_lines)

module_path = os.path.join(COMMON_DIR, "preprocessing.py")
with open(module_path, "w") as f:
    f.write(module_source)
print("Saved reusable module:", module_path)
print(f"  ({len(module_source)} characters, {module_source.count(chr(10)) + 1} lines)")

config = {
    "seed": SEED,
    "clahe_clip_limit": CLAHE_CLIP_LIMIT,
    "clahe_tile_grid_size": list(CLAHE_TILE_GRID_SIZE),
    "resize_size": RESIZE_SIZE,
    "crop_size": CROP_SIZE,
    "imagenet_mean": IMAGENET_MEAN,
    "imagenet_std": IMAGENET_STD,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "dataset_out_dir": DATASET_OUT_DIR,
    "results_out_dir": RESULTS_OUT_DIR,
}
config_path = os.path.join(COMMON_DIR, "preprocessing_config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print("Saved config:", config_path)

print("\nTo use in Notebook 3+:")
print("  import sys; sys.path.insert(0, COMMON_DIR)")
print("  from preprocessing import build_train_transform, build_eval_transform, GlaucomaDataset, get_dataloaders, compute_class_weights, denormalize")

## 17. Verify the saved module actually imports and works

Sanity-checks that `common/preprocessing.py` is not just written to disk but
is actually importable and produces the same behavior as the in-notebook
definitions -- this is what Notebook 3+ will rely on.

In [ ]:
if COMMON_DIR not in sys.path:
    sys.path.insert(0, COMMON_DIR)

import importlib
import preprocessing as _pp
importlib.reload(_pp)

_reloaded_train_transform = _pp.build_train_transform()
_reloaded_eval_transform = _pp.build_eval_transform()
_reloaded_ds = _pp.GlaucomaDataset(manifest_df, "test", DATASET_OUT_DIR, transform=_reloaded_eval_transform)

print("Imported preprocessing module from:", _pp.__file__)
print("Reloaded test dataset size:", len(_reloaded_ds))
assert len(_reloaded_ds) == EXPECTED_SPLIT["test"]

_img, _label = _reloaded_ds[0]
print("Sample from reloaded dataset -- image shape:", tuple(_img.shape), "label:", _label)
assert tuple(_img.shape) == (3, CROP_SIZE, CROP_SIZE)
print("\nPASS: common/preprocessing.py is importable and behaves identically to the in-notebook definitions.")

## 18. Final verification summary

Re-checks that `split_manifest.csv` was never modified (MD5 hash unchanged)
and that the raw dataset's file count/size are unchanged, then prints a
consolidated summary of everything verified in this notebook.

In [ ]:
split_manifest_md5_after = file_md5(split_manifest_path)
split_manifest_unchanged = (split_manifest_md5_before == split_manifest_md5_after)

raw_n_files_after, raw_bytes_after = raw_dataset_fingerprint(RAW_DATASET_ROOT)
raw_unchanged = (raw_n_files_after == raw_n_files_before) and (raw_bytes_after == raw_bytes_before)

print("=" * 60)
print("NOTEBOOK 2 -- FINAL VERIFICATION SUMMARY")
print("=" * 60)
print(f"split_manifest.csv MD5 unchanged : {split_manifest_unchanged} "
      f"({split_manifest_md5_before} -> {split_manifest_md5_after})")
print(f"Raw dataset unchanged (files/bytes): {raw_unchanged} "
      f"({raw_n_files_before}/{raw_bytes_before} -> {raw_n_files_after}/{raw_bytes_after})")
print(f"Dataset sizes match approved split : train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")
print(f"CLAHE params                       : clip_limit={CLAHE_CLIP_LIMIT}, tile_grid_size={CLAHE_TILE_GRID_SIZE}")
print(f"Train transform includes RandomVerticalFlip: "
      f"{any(isinstance(t, transforms.RandomVerticalFlip) for t in train_transform.transforms)} (must be False)")
print(f"Class weights (Normal, Glaucoma)   : {class_weights.tolist()}")
print(f"Reusable module saved at           : {module_path}")
print(f"Reusable config saved at           : {config_path}")
print(f"Figures saved at                   : {FIGURES_OUT_DIR}")
print("=" * 60)

assert split_manifest_unchanged, "split_manifest.csv was modified -- this must never happen in Notebook 2."
assert raw_unchanged, "Raw dataset changed -- this must never happen in Notebook 2."
print("\nALL CHECKS PASSED. Notebook 2 is verified and did not alter Notebook 1's split or the raw dataset.")